# Data Preprocessing Techniques — Complete Guide

## 📌 What is Data Preprocessing?

Data preprocessing is the process of cleaning and transforming raw data before using it in machine learning models.

Raw datasets often contain:
- missing values
- categorical text values
- features with different scales
- outliers
- irrelevant or weak features

If we train models directly on raw data, the model may perform poorly.

---

## 🎯 Goals of Preprocessing
- improve data quality
- make data suitable for machine learning
- reduce noise
- improve model accuracy
- make training more stable

---

## 📚 Techniques Covered
1. Data Loading
2. Basic Data Inspection
3. Missing Value Handling
4. Encoding Categorical Features
5. Feature Scaling
6. Outlier Detection and Removal
7. Feature Engineering
8. Feature Selection
9. Train-Test Split
10. Final Clean Dataset

---
## 📦 1. Imports & Data Loading

In this section, we:
- import required libraries
- load the dataset
- preview the data

These libraries help with:
- data handling using pandas
- numerical operations using numpy
- visualization using matplotlib and seaborn
- preprocessing using scikit-learn

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [2]:
# Replace this with your real dataset file
df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

target_col = "Survived"

print("✅ Dataset loaded successfully")
print("Shape of dataset:", df.shape)
df.head()

✅ Dataset loaded successfully
Shape of dataset: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
## 🔍 2. Basic Data Inspection

Before preprocessing, we need to understand:
- how many rows and columns are present
- data types of each column
- whether there are missing values
- basic statistics of numeric columns

This step helps us decide what preprocessing techniques are needed.

In [3]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
df.describe(include='all').T

Shape: (891, 12)

Column names:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PassengerId,891.0,NaN,NaN,NaN,446.0,257.353842,1.0,223.5,446.0,668.5,891.0
Survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
Pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
Name,891,891,"Braund, Mr. Owen Harris",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
SibSp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
Parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
Ticket,891,681,347082,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292


---
## 🧩 3. Missing Value Handling

### Why missing values are a problem
Machine learning models usually cannot handle empty values directly.

For example:
- if age is missing
- if income is missing
- if gender is missing

the model may fail or give poor results.

---

## Common strategies
### Numeric columns
- mean
- median

### Categorical columns
- most frequent value

---

## How it works
### Mean imputation
Replaces missing numeric values with the average of that column.

### Median imputation
Replaces missing numeric values with the middle value.
Useful when outliers are present.

### Most frequent imputation
Replaces missing categorical values with the most common category.

In [4]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

Numeric Columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [5]:
# Impute numeric columns using mean
num_imputer = SimpleImputer(strategy='mean')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# Impute categorical columns using most frequent value
if len(categorical_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

print("✅ Missing values handled")
print(df.isnull().sum())

✅ Missing values handled
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0
dtype: int64


### Key idea
After this step:
- numeric missing values are replaced with mean values
- categorical missing values are replaced with the most common value

This makes the dataset complete and usable for further preprocessing.

---
## 🔤 4. Encoding Categorical Features

### Why encoding is needed
Machine learning models work with numbers, not text.

Example:
- "Male", "Female"
- "Own", "Rent"
- "Graduate", "Not Graduate"

These text values must be converted into numeric format.

---

## Types of encoding

### 1. Label Encoding
Each category gets a number.

Example:
- Male → 0
- Female → 1

Useful when the feature has only two categories.

### 2. One-Hot Encoding
Creates separate binary columns for each category.

Example:
City:
- New York
- Chicago
- Dallas

Becomes:
- City_NewYork
- City_Chicago
- City_Dallas

Useful when categories have no natural order.

In [6]:
# Example: binary columns can use Label Encoding
label_encoded_df = df.copy()

binary_cols = []
for col in categorical_cols:
    if label_encoded_df[col].nunique() == 2:
        binary_cols.append(col)

print("Binary categorical columns:", binary_cols)

le = LabelEncoder()
for col in binary_cols:
    label_encoded_df[col] = le.fit_transform(label_encoded_df[col])

label_encoded_df.head()

Binary categorical columns: ['Sex']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1.0,0.0,3.0,"Braund, Mr. Owen Harris",1,22.0,1.0,0.0,A/5 21171,7.2500,B96 B98,S
1,2.0,1.0,1.0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1.0,0.0,PC 17599,71.2833,C85,C
2,3.0,1.0,3.0,"Heikkinen, Miss. Laina",0,26.0,0.0,0.0,STON/O2. 3101282,7.9250,B96 B98,S
3,4.0,1.0,1.0,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1.0,0.0,113803,53.1000,C123,S
4,5.0,0.0,3.0,"Allen, Mr. William Henry",1,35.0,0.0,0.0,373450,8.0500,B96 B98,S


In [7]:
# One-Hot Encoding for remaining categorical columns
remaining_cat_cols = [col for col in categorical_cols if col not in binary_cols]

encoded_df = pd.get_dummies(label_encoded_df, columns=remaining_cat_cols, drop_first=True)

print("✅ Categorical encoding completed")
print("New shape after encoding:", encoded_df.shape)
encoded_df.head()

✅ Categorical encoding completed
New shape after encoding: (891, 1726)


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,1.0,0.0,3.0,1,22.0,1.0,0.0,7.2500,False,False,...,False,False,False,False,False,False,False,False,False,True
1,2.0,1.0,1.0,0,38.0,1.0,0.0,71.2833,False,False,...,False,False,False,False,False,False,False,False,False,False
2,3.0,1.0,3.0,0,26.0,0.0,0.0,7.9250,False,False,...,False,False,False,False,False,False,False,False,False,True
3,4.0,1.0,1.0,0,35.0,1.0,0.0,53.1000,False,False,...,False,False,False,False,False,False,False,False,False,True
4,5.0,0.0,3.0,1,35.0,0.0,0.0,8.0500,False,False,...,False,False,False,False,False,False,False,False,False,True


### Key idea
- Label Encoding is compact and simple for binary categories.
- One-Hot Encoding is safer for multi-category columns because it does not create false numeric order.

After encoding, the dataset contains only numeric values, which makes it suitable for machine learning.

---
## 📏 5. Feature Scaling

### Why scaling is needed
Different features may have different ranges.

Example:
- Age: 20 to 60
- Income: 20,000 to 150,000
- Loan Amount: 1,000 to 50,000

Without scaling:
- large-value columns may dominate the model
- distance-based models like KNN and clustering can perform poorly

---

## Common scaling methods

### 1. Standardization
Formula:
\[
z = \frac{x - \mu}{\sigma}
\]

It transforms data so that:
- mean = 0
- standard deviation = 1

### 2. Min-Max Scaling
Formula:
\[
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
\]

It transforms values to the range:
- 0 to 1

In [8]:
# Create copies for scaling demonstration
standard_scaled_df = encoded_df.copy()
minmax_scaled_df = encoded_df.copy()

# Remove target temporarily if present
target_col = "target"  # change if needed

if target_col in encoded_df.columns:
    X_temp = encoded_df.drop(columns=[target_col])
    y_temp = encoded_df[target_col]
else:
    X_temp = encoded_df.copy()
    y_temp = None

# Standard Scaling
std_scaler = StandardScaler()
X_standard_scaled = pd.DataFrame(std_scaler.fit_transform(X_temp), columns=X_temp.columns)

print("✅ Standardization completed")
X_standard_scaled.head()

✅ Standardization completed


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,-1.730108,-0.789272,0.827377,0.737695,-0.592481,0.432793,-0.473674,-0.502445,-0.03352,-0.03352,...,-0.03352,-0.047431,-0.058124,-0.058124,-0.03352,-0.047431,-0.067153,-0.03352,-0.307562,0.615838
1,-1.726220,1.266990,-1.566107,-1.355574,0.638789,0.432793,-0.473674,0.786845,-0.03352,-0.03352,...,-0.03352,-0.047431,-0.058124,-0.058124,-0.03352,-0.047431,-0.067153,-0.03352,-0.307562,-1.623803
2,-1.722332,1.266990,0.827377,-1.355574,-0.284663,-0.474545,-0.473674,-0.488854,-0.03352,-0.03352,...,-0.03352,-0.047431,-0.058124,-0.058124,-0.03352,-0.047431,-0.067153,-0.03352,-0.307562,0.615838
3,-1.718444,1.266990,-1.566107,-1.355574,0.407926,0.432793,-0.473674,0.420730,-0.03352,-0.03352,...,-0.03352,-0.047431,-0.058124,-0.058124,-0.03352,-0.047431,-0.067153,-0.03352,-0.307562,0.615838
4,-1.714556,-0.789272,0.827377,0.737695,0.407926,-0.474545,-0.473674,-0.486337,-0.03352,-0.03352,...,-0.03352,-0.047431,-0.058124,-0.058124,-0.03352,-0.047431,-0.067153,-0.03352,-0.307562,0.615838


In [9]:
# Min-Max Scaling
minmax_scaler = MinMaxScaler()
X_minmax_scaled = pd.DataFrame(minmax_scaler.fit_transform(X_temp), columns=X_temp.columns)

print("✅ Min-Max Scaling completed")
X_minmax_scaled.head()

✅ Min-Max Scaling completed


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,0.000000,0.0,1.0,1.0,0.271174,0.125,0.0,0.014151,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.001124,1.0,0.0,0.0,0.472229,0.125,0.0,0.139136,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.002247,1.0,1.0,0.0,0.321438,0.000,0.0,0.015469,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.003371,1.0,0.0,0.0,0.434531,0.125,0.0,0.103644,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.004494,0.0,1.0,1.0,0.434531,0.000,0.0,0.015713,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### When to use which?
- StandardScaler: often used for Logistic Regression, SVM, KNN, PCA
- MinMaxScaler: useful when features must stay within a fixed range

Tree-based models like Decision Tree and Random Forest usually do not require scaling.

---
## 🚨 6. Outlier Detection and Removal

### What is an outlier?
An outlier is a value that is much smaller or larger than most other values.

Example:
- most salaries are between 30,000 and 100,000
- one salary is 5,000,000

That value may distort the model.

---

## IQR Method
The Interquartile Range (IQR) is:

\[
IQR = Q3 - Q1
\]

Where:
- Q1 = 25th percentile
- Q3 = 75th percentile

Outliers are values below:
\[
Q1 - 1.5 \times IQR
\]

or above:
\[
Q3 + 1.5 \times IQR
\]

In [10]:
# Use a numeric-only dataframe for outlier detection
numeric_only_df = X_standard_scaled.copy()

Q1 = numeric_only_df.quantile(0.25)
Q3 = numeric_only_df.quantile(0.75)
IQR = Q3 - Q1

outlier_mask = ((numeric_only_df < (Q1 - 1.5 * IQR)) | (numeric_only_df > (Q3 + 1.5 * IQR))).any(axis=1)

print("Number of outlier rows:", outlier_mask.sum())

X_no_outliers = numeric_only_df[~outlier_mask]

if y_temp is not None:
    y_no_outliers = y_temp[~outlier_mask]
else:
    y_no_outliers = None

print("Shape after outlier removal:", X_no_outliers.shape)

Number of outlier rows: 891
Shape after outlier removal: (0, 1726)


### Important note
Outlier removal should be done carefully.

Sometimes an outlier is:
- a true rare case
- important information

So we should not always remove outliers blindly.

---
## 🛠 7. Feature Engineering

### What is feature engineering?
Feature engineering means creating new useful columns from existing columns.

Example:
- BMI from weight and height
- income per family member
- loan-to-income ratio

This can improve model performance because the new feature may represent the problem better than raw columns.

In [11]:
feature_engineered_df = df.copy()

# Example feature engineering
# Only create if both columns exist
if 'loan_amnt' in feature_engineered_df.columns and 'person_income' in feature_engineered_df.columns:
    feature_engineered_df['loan_to_income_ratio'] = feature_engineered_df['loan_amnt'] / (feature_engineered_df['person_income'] + 1)
    print("✅ Created feature: loan_to_income_ratio")

feature_engineered_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1.0,0.0,3.0,"Braund, Mr. Owen Harris",male,22.0,1.0,0.0,A/5 21171,7.2500,B96 B98,S
1,2.0,1.0,1.0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC 17599,71.2833,C85,C
2,3.0,1.0,3.0,"Heikkinen, Miss. Laina",female,26.0,0.0,0.0,STON/O2. 3101282,7.9250,B96 B98,S
3,4.0,1.0,1.0,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0.0,113803,53.1000,C123,S
4,5.0,0.0,3.0,"Allen, Mr. William Henry",male,35.0,0.0,0.0,373450,8.0500,B96 B98,S


### Why this helps
A model may learn better from a meaningful ratio than from raw values separately.

For example:
- a loan amount of 20,000 may be safe for someone with high income
- the same loan amount may be risky for someone with low income

The ratio captures that relationship directly.

---
## 🎯 8. Feature Selection

### Why feature selection is useful
Not every feature helps the model.

Some features may be:
- irrelevant
- redundant
- noisy

Removing weak features can:
- simplify the model
- reduce overfitting
- improve training speed

---

## SelectKBest
This method selects the top K features based on statistical scores.

In [16]:
target_col = "Survived"
if target_col in encoded_df.columns:
    X_fs = encoded_df.drop(columns=[target_col])
    y_fs = encoded_df[target_col]

    selector = SelectKBest(score_func=f_classif, k=min(5, X_fs.shape[1]))
    X_selected = selector.fit_transform(X_fs, y_fs)

    selected_features = X_fs.columns[selector.get_support()]
    print("✅ Selected features:")
    print(selected_features.tolist())
else:
    print("⚠️ Target column not found. Feature selection skipped.")

✅ Selected features:
['Pclass', 'Sex', 'Fare', 'Cabin_B96 B98', 'Embarked_S']


### How it works
SelectKBest calculates a score for each feature and keeps only the top-scoring ones.

This helps focus the model on the most informative inputs.

---
## ✂️ 9. Train-Test Split

### Why split the data?
We need:
- training data → for learning
- testing data → for evaluation

This helps us check whether the model generalizes well to unseen data.

In [17]:
if target_col in encoded_df.columns:
    X_final = encoded_df.drop(columns=[target_col])
    y_final = encoded_df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X_final, y_final, test_size=0.2, random_state=42
    )

    print("✅ Train-Test Split Completed")
    print("X_train shape:", X_train.shape)
    print("X_test shape :", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape :", y_test.shape)
else:
    print("⚠️ Target column not found. Train-test split skipped.")

✅ Train-Test Split Completed
X_train shape: (712, 1725)
X_test shape : (179, 1725)
y_train shape: (712,)
y_test shape : (179,)


---
## 🧱 10. Preprocessing Pipeline

### Why use a pipeline?
A pipeline allows us to:
- automate preprocessing steps
- avoid repeated code
- apply the same steps consistently to train and test data

This is the professional and industry-style way to preprocess data.

In [18]:
# Re-create from original df
df_pipeline = pd.read_csv("loan_data.csv")

if target_col in df_pipeline.columns:
    X_pipe = df_pipeline.drop(columns=[target_col])
    y_pipe = df_pipeline[target_col]
else:
    X_pipe = df_pipeline.copy()
    y_pipe = None

num_cols_pipe = X_pipe.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols_pipe = X_pipe.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols_pipe),
    ('cat', categorical_transformer, cat_cols_pipe)
])

X_processed = preprocessor.fit_transform(X_pipe)

print("✅ Pipeline preprocessing completed")
print("Processed feature shape:", X_processed.shape)

✅ Pipeline preprocessing completed
Processed feature shape: (9578, 20)


### What happened here?
- numeric columns were imputed and scaled
- categorical columns were imputed and one-hot encoded
- all preprocessing was done in one structured workflow

This is the best way to build reusable ML code.

---
## 📌 11. Final Conclusion

In this notebook, we performed the most important preprocessing steps used in machine learning:

- handled missing values
- encoded categorical features
- scaled numeric features
- detected and removed outliers
- created new features
- selected important features
- split the dataset
- built a preprocessing pipeline

These steps help convert raw data into clean, machine-learning-ready data.

In [19]:
print("✅ Preprocessing notebook completed successfully")

✅ Preprocessing notebook completed successfully
